
# Calculating selected-week state boundaries

This notebook calculates the discretisation thresholds for the DBN input nodes using the selected target-week implementation scope. Since the current DBN implementation focuses on one selected target week, the input-node thresholds are calculated for the same selected-week scope.

Default selected target week: July 8th 00:00 to July 15th 00:00

By default, thresholds are calculated from the **model scope**, meaning all connected hourly Dataset 4 rows for sessions whose connection windows intersect the selected target week. This matches the current DBN implementation because shifting is allowed within each session's full connection window $\mathcal{T}_i$, not only inside the calendar target-week timestamps.


In [19]:

from pathlib import Path
import pandas as pd
import numpy as np

# =============================================================================
# Selected-week state-boundary calculation
# =============================================================================
# Purpose:
# This notebook calculates the discretisation thresholds for the selected
# target-week
#
# Selected target week used in the DBN implementation:
# July 8th 00:00 to July 15th 00:00
#
# Important modelling choice:
# The default threshold scope below is "model_scope". This matches the DBN code:
# it uses all Dataset 4 connected hourly rows for sessions whose connection
# windows intersect the selected target week. This preserves the thesis logic
# that charging may be shifted within the full connection window T_i.
#
# For transparency, the notebook also calculates a comparison based on calendar
# rows strictly inside the selected week only.

# ------------------------------------------------------------
# 1. File paths and configuration
# ------------------------------------------------------------

DATASET1_NAME = "/Users/yadkhan/Downloads/Case study files/Dataset1_charging_reports.csv"
DATASET2_NAME = "/Users/yadkhan/Downloads/Case study files/Dataset2_user_predictions.csv"
DATASET3_NAME = "/Users/yadkhan/Downloads/Case study files/Dataset3_session_predictions.csv"
DATASET4_NAME = "/Users/yadkhan/Downloads/Case study files/Dataset4_hourly_predictions.csv"

LOCATION = "TRO_R"
TARGET_WEEK_START = pd.Timestamp("2019-07-08 00:00:00")
TARGET_WEEK_DAYS = 7
TARGET_WEEK_END = TARGET_WEEK_START + pd.Timedelta(days=TARGET_WEEK_DAYS)

TARGET_START_HOUR = 16 # Evening hours start from 16:00
TARGET_END_HOUR_EXCLUSIVE = 22 # Evening hours end at 22:00 (up to and including 21:00)

# Choose the threshold basis used for the final recommended thresholds.
# - "model_scope": rows used by the selected-week DBN implementation, i.e.
#   all connected hourly rows in Dataset 4 for sessions intersecting the target week.
# - "calendar_week": only Dataset 4 rows whose date_from is inside the selected calendar week.
THRESHOLD_SCOPE_FOR_FINAL_VALUES = "model_scope"


def resolve_csv(filename: str) -> Path:
    """Find a CSV file robustly"""
    direct = DATA_DIR / filename
    if direct.exists():
        return direct
    matches = sorted(DATA_DIR.glob(Path(filename).stem + "*.csv"))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Could not find {filename} in {DATA_DIR.resolve()}")


def read_case_csv(path: Path) -> pd.DataFrame:
    """Read the Sørensen et al. CSV format: semicolon-separated with comma decimals."""
    return pd.read_csv(path, sep=";", decimal=",", na_values=["NA", ""], keep_default_na=True)

# ------------------------------------------------------------
# 2. Load datasets
# ------------------------------------------------------------

dataset1_path = resolve_csv(DATASET1_NAME)
dataset2_path = resolve_csv(DATASET2_NAME)
dataset3_path = resolve_csv(DATASET3_NAME)
dataset4_path = resolve_csv(DATASET4_NAME)

# Dataset 2 is loaded for completeness, but the current DBN input-node thresholds
# are calculated from Dataset 4 after filtering with Dataset 1 and Dataset 3.
d1 = read_case_csv(dataset1_path)
d2 = read_case_csv(dataset2_path)
d3 = read_case_csv(dataset3_path)
d4 = read_case_csv(dataset4_path)

# Convert date columns used to define session windows and hourly time.
d1["plugin_time"] = pd.to_datetime(d1["plugin_time"], errors="coerce")
d1["plugout_time"] = pd.to_datetime(d1["plugout_time"], errors="coerce")
d4["date_from"] = pd.to_datetime(d4["date_from"], errors="coerce")

# ------------------------------------------------------------
# 3. Select the same target-week scope as the DBN implementation
# ------------------------------------------------------------
# Sessions are selected when their physical connection window intersects the
# target week and the complete connection window stays inside 2019. This mirrors
# the current DBN implementation and avoids 2018/2020 boundary cases.

session_level = d1[d1["location"].eq(LOCATION)].merge(
    d3[["session_id", "charging_time", "idle_time", "non_flex_session"]],
    on="session_id",
    how="left",
)

sessions_week = session_level[
    (session_level["plugin_time"] < TARGET_WEEK_END)
    & (session_level["plugout_time"] >= TARGET_WEEK_START)
    & (session_level["plugin_time"] >= pd.Timestamp("2019-01-01 00:00:00"))
    & (session_level["plugout_time"] < pd.Timestamp("2020-01-01 00:00:00"))
].copy()

required_session_cols = ["plugin_time", "plugout_time", "connection_time", "charging_time", "idle_time"]
missing_required = sessions_week[required_session_cols].isna().sum()
if missing_required.sum() > 0:
    raise ValueError(
        "Selected target week contains missing required session-level values: "
        f"{missing_required.to_dict()}"
    )

# Chapter 6 treats sessions with idle_time >= 1 hour as flexible. The current DBN
# code also requires the Dataset 3 non_flex_session flag to be missing.
sessions_week["is_flexible"] = sessions_week["idle_time"].ge(1.0) & sessions_week["non_flex_session"].isna()

# Dataset 4 rows for the selected sessions. We keep all connected hourly rows for
# those sessions inside 2019, because shifting is allowed across T_i, not only
# inside the weekday-evening target hours.
hourly_model_scope = d4[d4["session_id"].isin(sessions_week["session_id"])].copy()
hourly_model_scope = hourly_model_scope[
    (hourly_model_scope["date_from"] >= pd.Timestamp("2019-01-01 00:00:00"))
    & (hourly_model_scope["date_from"] < pd.Timestamp("2020-01-01 00:00:00"))
].copy()

if hourly_model_scope[["user_id", "session_id", "date_from", "energy_connected_i"]].isna().any().any():
    raise ValueError("Selected hourly scope has missing identifiers, timestamps, or energy_connected_i.")

# Appendix A treats missing observed charging and idle capacity as zero for the
# zero state of the corresponding input nodes.
hourly_model_scope["energy_charged_raw"] = hourly_model_scope["energy_charged_i"]
hourly_model_scope["energy_idle_raw"] = hourly_model_scope["energy_idle_i"]
hourly_model_scope["energy_charged_i"] = hourly_model_scope["energy_charged_i"].fillna(0.0)
hourly_model_scope["energy_idle_i"] = hourly_model_scope["energy_idle_i"].fillna(0.0)

# Comparison scope: rows strictly inside the selected calendar week.
hourly_calendar_week = hourly_model_scope[
    (hourly_model_scope["date_from"] >= TARGET_WEEK_START)
    & (hourly_model_scope["date_from"] < TARGET_WEEK_END)
].copy()

# ------------------------------------------------------------
# 4. Helper functions for selected-week boundaries
# ------------------------------------------------------------

def positive_quantiles(series: pd.Series, probs=(0.33, 0.67)) -> pd.Series:
    """
    Calculate quantile boundaries over positive values only.

    The zero state is handled separately, so zeros and missing values are not
    included when calculating the low/medium/high boundaries.
    """
    positive_values = series.fillna(0.0)[series.fillna(0.0) > 0]
    if positive_values.empty:
        raise ValueError("Cannot calculate positive quantiles because the series has no positive values.")
    return positive_values.quantile(list(probs), interpolation="linear")


def calculate_energy_thresholds(df: pd.DataFrame, scope_name: str) -> pd.DataFrame:
    """Calculate q33/q67 thresholds for the three DBN input nodes."""
    mapping = {
        "e_obs": "energy_charged_i",
        "e_idle": "energy_idle_i",
        "e_conn": "energy_connected_i",
    }
    rows = []
    for node, col in mapping.items():
        q = positive_quantiles(df[col], probs=(0.33, 0.67))
        rows.append({
            "scope": scope_name,
            "node": node,
            "dataset_column": col,
            "positive_value_count": int((df[col].fillna(0.0) > 0).sum()),
            "q33": float(q.loc[0.33]),
            "q67": float(q.loc[0.67]),
            "rounded_q33": round(float(q.loc[0.33]), 2),
            "rounded_q67": round(float(q.loc[0.67]), 2),
        })
    return pd.DataFrame(rows)

# ------------------------------------------------------------
# 5. Calculate selected-week input-node thresholds
# ------------------------------------------------------------

thresholds_model_scope = calculate_energy_thresholds(hourly_model_scope, "model_scope")
thresholds_calendar_week = calculate_energy_thresholds(hourly_calendar_week, "calendar_week")
thresholds_all = pd.concat([thresholds_model_scope, thresholds_calendar_week], ignore_index=True)

final_thresholds = thresholds_all[thresholds_all["scope"].eq(THRESHOLD_SCOPE_FOR_FINAL_VALUES)].copy()

# ------------------------------------------------------------
# 6. Calculate selected-week C90 for the target evidence
# ------------------------------------------------------------
# This follows the DBN implementation: C90 is calculated from weekday-evening
# target hours in the selected week, including zero-load target hours if present.

all_week_hours = pd.date_range(
    TARGET_WEEK_START,
    TARGET_WEEK_END - pd.Timedelta(hours=1),
    freq="h",
)

target_hours = pd.DatetimeIndex([
    ts for ts in all_week_hours
    if ts.weekday() < 5 and TARGET_START_HOUR <= ts.hour < TARGET_END_HOUR_EXCLUSIVE
])

week_rows = hourly_calendar_week.copy()
load_by_hour = week_rows.groupby("date_from")["energy_charged_i"].sum()
target_loads = pd.Series(
    [float(load_by_hour.get(ts, 0.0)) for ts in target_hours],
    index=target_hours,
    name="L_obs_t",
)

C90 = float(target_loads.quantile(0.90, interpolation="linear"))
exceeded_hours = target_loads[target_loads > C90 + 1e-12]

# ------------------------------------------------------------
# 7. Validation check for connected energy relation
# ------------------------------------------------------------
# This verifies the expected relation energy_connected_i ≈ energy_charged_i + energy_idle_i
# after missing observed charging and idle capacity are mapped to zero.

connection_check = (
    hourly_model_scope["energy_connected_i"]
    - (hourly_model_scope["energy_charged_i"] + hourly_model_scope["energy_idle_i"])
).abs()
max_connection_difference = float(connection_check.max())

# ------------------------------------------------------------
# 8. Save threshold outputs
# ------------------------------------------------------------

output_dir = DATA_DIR / "selected_week_state_boundaries"
output_dir.mkdir(parents=True, exist_ok=True)

thresholds_all.to_csv(output_dir / "selected_week_input_node_thresholds_comparison.csv", index=False)
final_thresholds.to_csv(output_dir / "selected_week_input_node_thresholds_final.csv", index=False)
target_loads.rename("observed_target_load_kwh").to_frame().to_csv(output_dir / "selected_week_target_loads_for_C90.csv")

# ------------------------------------------------------------
# 9. Print results
# ------------------------------------------------------------

print("Selected target-week scope")
print("--------------------------")
print(f"Location: {LOCATION}")
print(f"Target week start: {TARGET_WEEK_START}")
print(f"Target week end:   {TARGET_WEEK_END}")
print(f"Threshold scope used for final values: {THRESHOLD_SCOPE_FOR_FINAL_VALUES}")
print(f"Sessions intersecting target week: {len(sessions_week)}")
print(f"Flexible sessions: {int(sessions_week['is_flexible'].sum())}")
print(f"Hourly rows in model scope: {len(hourly_model_scope)}")
print(f"Hourly rows strictly inside calendar week: {len(hourly_calendar_week)}")
print(f"Hourly rows outside calendar week but inside selected sessions' T_i: {len(hourly_model_scope) - len(hourly_calendar_week)}")
print(f"Target weekday-evening hours: {len(target_hours)}")
print()

print("Validation check")
print("----------------")
print(f"Max |energy_connected_i - (energy_charged_i + energy_idle_i)|: {max_connection_difference:.10f}")
print()

print("Selected-week input-node discretisation thresholds")
print("---------------------------------------------------")
print(thresholds_all.to_string(index=False))
print()

print("Final values to use in the DBN implementation")
print("----------------------------------------------")
for _, row in final_thresholds.iterrows():
    print(f"{row['node']}: q33 = {row['q33']:.6f}, q67 = {row['q67']:.6f}  "
          f"(rounded: {row['rounded_q33']:.2f}, {row['rounded_q67']:.2f})")
print(f"C90 for selected target week: {C90:.6f} kWh  (rounded: {C90:.2f})")
print()

print("Copy-paste dictionary for the DBN code")
print("--------------------------------------")
print("ENERGY_STATE_THRESHOLDS = {")
for _, row in final_thresholds.iterrows():
    print(f"    \"{row['node']}\": {{\"q33\": {row['q33']:.12g}, \"q67\": {row['q67']:.12g}}},")
print("}")
print()

print("Target-hour exceedances under selected-week C90")
print("-----------------------------------------------")
if exceeded_hours.empty:
    print("No target hours exceed C90.")
else:
    for ts, value in exceeded_hours.items():
        print(f"{ts}: L_obs = {value:.6f} kWh, exceeds C90 by {value - C90:.6f} kWh")
print()

print("Output files written to:")
print(output_dir)


Selected target-week scope
--------------------------
Location: TRO_R
Target week start: 2019-07-08 00:00:00
Target week end:   2019-07-15 00:00:00
Threshold scope used for final values: model_scope
Sessions intersecting target week: 39
Flexible sessions: 29
Hourly rows in model scope: 455
Hourly rows strictly inside calendar week: 405
Hourly rows outside calendar week but inside selected sessions' T_i: 50
Target weekday-evening hours: 30

Validation check
----------------
Max |energy_connected_i - (energy_charged_i + energy_idle_i)|: 0.0000000000

Selected-week input-node discretisation thresholds
---------------------------------------------------
        scope   node     dataset_column  positive_value_count      q33      q67  rounded_q33  rounded_q67
  model_scope  e_obs   energy_charged_i                   167 3.242339 4.521205         3.24         4.52
  model_scope e_idle      energy_idle_i                   327 3.579171 6.260996         3.58         6.26
  model_scope e_conn ene